<a href="https://colab.research.google.com/github/abood-W/Fine-Tuning/blob/main/fineTunning_a_model_using_XLNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install cleantext
!pip install evaluate
import pandas as pd
import numpy as np
from cleantext import clean
import re
from transformers import XLNetTokenizer ,XLNetForSequenceClassification,TrainingArguments,Trainer,pipeline
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import datasets
import evaluate
import random

In [ ]:
data_train = pd.read_csv('/content/emotion-labels-train.csv')
data_test = pd.read_csv('/content/emotion-labels-test.csv')
data_val = pd.read_csv('/content/emotion-labels-val.csv')

In [ ]:
data_train['label'].unique()

In [ ]:
data_train.head()

In [ ]:
data = pd.concat([data_train,data_test],ignore_index=True)

In [ ]:
data['text_clean']=data['text'].apply(lambda x : clean(x))

In [ ]:
data['text_clean']=data['text_clean'].apply(lambda x : re.sub('@[^\s]+', '', x))

### Remove Emojis
Now, let's remove any remaining emojis from the `text_clean` column using a regular expression.

In [ ]:
def remove_emojis(text):
    emoji_pattern = re.compile(
        "["  # Start character class
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F300-\U0001F5FF"  # symbols & pictographs
        "\U0001F680-\U0001F6FF"  # transport & map symbols
        "\U0001F1E0-\U0001F1FF"  # flags (iOS)
        "\U00002702-\U000027B0"  # Dingbats
        "\U000024C2-\U0001F251"  # Enclosed CJK Letters and Months
        "]+"
    )
    return emoji_pattern.sub(r'', text)

data['text_clean'] = data['text_clean'].apply(remove_emojis)

In [ ]:
def clean_text_further(text):
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Convert to lowercase
    text = text.lower()
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

data['text_clean'] = data['text_clean'].apply(clean_text_further)

In [ ]:
display(data.head())

In [ ]:
data['label'].value_counts().plot(kind='bar')

In [ ]:
g=data.groupby('label')
data=pd.DataFrame(g.apply(lambda x: x.sample(g.size().min()).reset_index(drop=True)))

In [ ]:
data['label'].value_counts().plot(kind='bar')

In [ ]:
data['label_int'] = LabelEncoder().fit_transform(data['label'])


In [ ]:
NUM_LABELS = 4

In [ ]:
train_split , test_split = train_test_split(data,train_size=0.8)
train_split ,val_split = train_test_split(train_split,train_size=0.9)

In [ ]:
print(len(train_split))
print(len(val_split))
print(len(test_split))

In [ ]:
train_df = pd.DataFrame({
    "label":train_split.label_int.values,
    "text":train_split.text_clean.values
})

test_df = pd.DataFrame({
    "label":test_split.label_int.values,
    "text":test_split.text_clean.values
})

In [ ]:
train_df = datasets.Dataset.from_dict(train_df)
test_df = datasets.Dataset.from_dict(test_df)

In [ ]:
datasets_dict = datasets.DatasetDict({"train":train_df,"test":test_df})

In [ ]:
datasets_dict

In [ ]:
tokenizer = XLNetTokenizer.from_pretrained("xlnet-base-cased")

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True,max_length=128)

In [ ]:
tokenized_datasets = datasets_dict.map(tokenize_function, batched=True)

In [ ]:
tokenized_datasets

In [ ]:
print(tokenized_datasets['train']['input_ids'][0])

In [ ]:
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(100))
small_eval_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(100))

In [ ]:
model = XLNetForSequenceClassification.from_pretrained(
    "xlnet-base-cased",
    num_labels=NUM_LABELS ,
    id2label={0: "anger", 1: "fear", 2: "joy", 3: "sadness"},
    label2id={"anger": 0, "fear": 1, "joy": 2, "sadness": 3},
)


In [ ]:
metric = evaluate.load("accuracy")

In [ ]:
def comput_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
training_arg = TrainingArguments(output_dir="test_trainer",eval_strategy="epoch",num_train_epochs=3)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_arg,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=comput_metrics,
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
model.save_pretrained('fine_tuned_model')

In [ ]:
fine_tuned_moel = XLNetForSequenceClassification.from_pretrained('fine_tuned_model')

In [ ]:
clf = pipeline("text-classification" ,fine_tuned_moel,tokenizer=tokenizer)

In [ ]:
rand_int=random.randint(0,len(val_split)-1)
print(val_split['text_clean'].iloc[rand_int])
answer=clf(val_split['text_clean'].iloc[rand_int])
print(answer)